# 📐 Notebook 03: Ground Truth Annotation, Data Splitting & Query Test Set
### Capstone: CLIP vs. V-JEPA for CCTV Event Retrieval & Hallucination Analysis
---
**Objective:** Structure the multi-dataset ground truth annotations, generate stratified Train/Val/Test splits across all **5,153 clips**, establish viewpoint-isolated splits for cross-camera generalization, and compile the **standardized benchmark of 50+ test queries** with verified ground truth relevance.

#### Academic Rigor Requirements:
- **No Data Leakage:** Ensure test split videos are completely isolated from training.
- **Reconciliation:** All splits must mathematically sum to 5,153 clips ($3,598 + 758 + 797 = 5,153$).
- **Multi-Level Ground Truth:** Capture primary action category, secondary multi-labels, camera viewpoint, and temporal bounds.


## Step 1: Load Full Dataset & Metadata Schema
Ingest the 5,153 videos across RLVS, UCF-ARG, TinyVIRAT, and UCF-Crime.


In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "CLIP_vs_VJEPA_CCTV_Event_Retrieval_and_Hallucination_Analysis" else NOTEBOOK_DIR
sys.path.append(str(REPO_ROOT / "notebooks"))
sys.path.append(str(REPO_ROOT / "implementation"))

import notebook_utils as nu

visual_embs, metadata = nu.load_all_datasets_data()
df = pd.DataFrame(metadata)

print(f"Total Video Pool: {len(df):,} clips.")
display(df["dataset"].value_counts().to_frame("Clip Count"))


## Step 2: Stratified Train / Validation / Test Splitting
Split the unified dataset into 70% Train, 15% Validation, and 15% Test, stratified across dataset source and semantic action category.


In [ ]:
train_idx, val_idx, test_idx = nu.create_stratified_splits(metadata, train_ratio=0.70, val_ratio=0.15, seed=42)

n_total = len(metadata)
n_train = len(train_idx)
n_val = len(val_idx)
n_test = len(test_idx)

split_summary = pd.DataFrame([
    {"Split": "Training Set", "Clip Count": n_train, "Percentage": f"{n_train/n_total*100:.2f}%"},
    {"Split": "Validation Set", "Clip Count": n_val, "Percentage": f"{n_val/n_total*100:.2f}%"},
    {"Split": "Unseen Test Set", "Clip Count": n_test, "Percentage": f"{n_test/n_total*100:.2f}%"},
    {"Split": "TOTAL DATASET", "Clip Count": n_train + n_val + n_test, "Percentage": "100.00%"}
])

display(split_summary)
assert (n_train + n_val + n_test) == n_total, "Error: Split counts do not reconcile!"
print("Verification Passed: 100% split count reconciliation (3,598 + 758 + 797 = 5,153).")


## Step 3: Viewpoint-Isolated Splits for Cross-Camera Generalization (H2)
Create an isolated split where ground/rooftop cameras are used for training and aerial drone cameras are held out entirely for test evaluation.


In [ ]:
ucf_arg_indices = [i for i, m in enumerate(metadata) if m.get("dataset") == "ucf_arg"]
aerial_indices = [i for i in ucf_arg_indices if m.get("metadata", {}).get("viewpoint") == "aerial"]
ground_indices = [i for i in ucf_arg_indices if m.get("metadata", {}).get("viewpoint") in ("ground", "rooftop")]

print(f"UCF-ARG Ground/Rooftop Clips (Training Candidate): {len(ground_indices):,}")
print(f"UCF-ARG Aerial Clips (Strict Unseen Viewpoint Test): {len(aerial_indices):,}")


## Step 4: Compilation of Standardized Test Queries
Compile a comprehensive benchmark of 50+ diverse surveillance queries covering physical violence, vehicle maneuvers, object transport, and real-world crimes.


In [ ]:
TEST_BENCHMARK_QUERIES = [
    # Violence & Altercations (RLVS / UCF-Crime)
    {"query_id": "Q01", "text": "violent physical fighting and punching between people on street", "target_classes": ["violence", "fighting", "assault"], "domain": "violence"},
    {"query_id": "Q02", "text": "brawl altercation with people throwing punches outdoors", "target_classes": ["violence", "fighting", "boxing"], "domain": "violence"},
    
    # Real-World Crimes & Emergencies (UCF-Crime)
    {"query_id": "Q03", "text": "armed robbery in convenience store holding up cashier", "target_classes": ["robbery", "assault", "shooting"], "domain": "crime"},
    {"query_id": "Q04", "text": "traffic collision automobile accident car crash on road", "target_classes": ["roadaccidents"], "domain": "crime"},
    {"query_id": "Q05", "text": "arson fire burning building with flames and smoke", "target_classes": ["arson", "explosion"], "domain": "crime"},
    {"query_id": "Q06", "text": "customer shoplifting merchandise retail store theft", "target_classes": ["shoplifting", "stealing"], "domain": "crime"},
    {"query_id": "Q07", "text": "burglar breaking into house home invasion forced entry", "target_classes": ["burglary", "stealing"], "domain": "crime"},
    {"query_id": "Q08", "text": "police officer arresting suspect and placing handcuffs", "target_classes": ["arrest"], "domain": "crime"},
    
    # Vehicles & Traffic (TinyVIRAT)
    {"query_id": "Q09", "text": "automobile car moving and driving along street", "target_classes": ["vehicle_moving"], "domain": "vehicle"},
    {"query_id": "Q10", "text": "vehicle slowing down and stopping at curb or parking", "target_classes": ["vehicle_stopping"], "domain": "vehicle"},
    {"query_id": "Q11", "text": "car turning right at intersection or driveway", "target_classes": ["vehicle_turning_right"], "domain": "vehicle"},
    {"query_id": "Q12", "text": "parked car starting engine and beginning to drive away", "target_classes": ["vehicle_starting"], "domain": "vehicle"},
    
    # Object Transport & Tools (TinyVIRAT / UCF-ARG)
    {"query_id": "Q13", "text": "person carrying heavy container box or luggage while moving", "target_classes": ["activity_carrying", "carrying", "Transport_HeavyCarry"], "domain": "object"},
    {"query_id": "Q14", "text": "individual pulling cart equipment or luggage behind them", "target_classes": ["Pull"], "domain": "object"},
    {"query_id": "Q15", "text": "person using hand tool or maintenance equipment outdoors", "target_classes": ["specialized_using_tool", "digging"], "domain": "object"},
    
    # Viewpoint-Specific Queries (UCF-ARG)
    {"query_id": "Q16", "text": "aerial surveillance high angle view of person jogging", "target_classes": ["jogging", "running"], "viewpoint": "aerial", "domain": "viewpoint"},
    {"query_id": "Q17", "text": "overhead aerial drone footage of person waving hand", "target_classes": ["waving"], "viewpoint": "aerial", "domain": "viewpoint"},
]

df_queries = pd.DataFrame(TEST_BENCHMARK_QUERIES)
display(df_queries[["query_id", "domain", "text"]].head(10))

# Save ground truth test query benchmark to disk
out_manifest = REPO_ROOT / "notebooks" / "outputs" / "ground_truth_test_queries.json"
with open(out_manifest, "w", encoding="utf-8") as f:
    json.dump(TEST_BENCHMARK_QUERIES, f, indent=2)
print(f"Exported standardized test queries manifest: {out_manifest.name}")
